# <center> 数据采集

### 1. 导入包

In [1]:
import os
import sys
import time
import queue
import threading
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import random
import Arm_Lib

### 2. 定义全局变量

In [2]:
cfg_dir = "config/"
XYT_cfg_dir = os.path.join(cfg_dir, "XYT_config.txt") # XYT参数保存文件
data_idx_cfg_dir = os.path.join(cfg_dir, "data_collect_idx.txt")
dp_cfg_dir = os.path.join(cfg_dir, "dp.bin")
CAMERA_FRAME_WIDTH = 640  # 图像显示的宽
CAMERA_FRAME_HEIGHT = 480 # 图像显示的高

IMAGE = False 
CALIB = True # 是否进行透视变换
FLAG = True  # 该参数决定是否将捕获后的图像保存在队列中

q = queue.Queue(maxsize=10) # 定义队列最大长度

# 定义图像保存路径
uuid_str = time.strftime("%Y-%m-%d-%H_%M_%S", time.localtime())
tmp_folder_name ='Images-%s' % uuid_str
os.makedirs(tmp_folder_name, exist_ok=True)

### 3. 初始化机械臂

In [3]:
def read_XYT(rf_path):
    dict = {}
    rf = open(rf_path, "r+")
    for line in rf.readlines():
        index = line.find('=')
        dict[line[:index]] = line[index + 1:]
    xy = [int(dict['x']), int(dict['y'])]
    rf.flush()
    return xy

In [4]:
xy = read_XYT(XYT_cfg_dir)
joints_0 = [xy[0], xy[1], 0, 0,90, 30] 
arm = Arm_Lib.Arm_Device()
arm.Arm_serial_servo_write6_array(joints_0, 1000)

### 4. 定义透视变换方法

In [5]:
def Perspective_transform(dp, image):
    if len(dp) != 4: 
        return
    upper_left = []
    lower_left = []
    lower_right = []
    upper_right = []
    for i in range(len(dp)):
        if dp[i][0] < 320 and dp[i][1] < 240:
            upper_left=dp[i]
        if dp[i][0] < 320 and dp[i][1] > 240:
            lower_left=dp[i]
        if dp[i][0] > 320 and dp[i][1] > 240:
            lower_right=dp[i]
        if dp[i][0] > 320 and dp[i][1] < 240:
            upper_right=dp[i]
    # 原图中的四个顶点
    pts1 = np.float32([upper_left, lower_left, lower_right, upper_right])
    # 变换后的四个顶点
    pts2 = np.float32([[0, 0], [0, 480], [640, 480], [640, 0]])
    # 根据四对对应点计算透视变换。
    M = cv2.getPerspectiveTransform(pts1, pts2)
    # 将透视变换应用于图像。
    Transform_img = cv2.warpPerspective(image, M, (640, 480))
    return Transform_img

### 5. 定义获取摄像头画面并实时显示的线程

In [6]:
imgbox = widgets.Image(format='jpg', height=CAMERA_FRAME_HEIGHT, width=CAMERA_FRAME_WIDTH)

In [7]:
# 获取摄像头画面并实时显示 线程函数
def capture_video():
    display(imgbox)
    
    capture = cv2.VideoCapture(0)
    
    while capture.isOpened():
        _, image = capture.read()
        if image is None:
            print("Get memory from camera failed")
            break
        
        if CALIB: 
            dp = np.fromfile(dp_cfg_dir, dtype=np.int32) # 读取透视变换参数并将其转化为NumPy数组
            dp = dp.reshape(4,2)
            image = Perspective_transform(dp, image) # 将透视变换应用于图像
            
        imgbox.value = cv2.imencode('.jpg', image)[1].tobytes() # 编码透视变换后的图像   
        global IMAGE  # 全局变量IMAGE决定是否将捕获的图像保存在队列（q）中。如果队列已满，则进程将等待，直到有可用的空间。

        if IMAGE:
            if q.full():
                continue
            else:
                q.put(image)
            IMAGE = False

### 6. 定义“保存图片”按钮线程

In [8]:
# 捕获摄像头画面 IMAGE 控制全局变量
def cap_image(change):
    global IMAGE
    IMAGE = True

# 捕获摄像头画面按钮显示 线程函数
def button_control():
    # 创建按钮
    button_layout = widgets.Layout(width='100px', height='80px', align_self='center')
    get_image_button = widgets.Button(description='保存图片', button_style='danger', layout=button_layout)
    
    # 显示按钮
    middle_box = widgets.HBox([get_image_button], layout=widgets.Layout(align_self='center'))
    
    # link buttons to actions
    get_image_button.on_click(cap_image) 
    display(middle_box)

### 7. 定义保存图片线程

In [9]:
# 保存图片到文件 线程函数
def save_image():
    while FLAG:
        img = q.get()
        with open(data_idx_cfg_dir, "r") as f:
            contents = f.readline()
            count = int(contents)
        uuid_str = str(count)
        print(uuid_str)
        tmp_file_name ='%s.jpg' % uuid_str
        print(os.path.join(tmp_folder_name, tmp_file_name))
        print(img.shape)
        print(type(img))
        if not cv2.imwrite(os.path.join(tmp_folder_name, tmp_file_name), img):
            raise Exception("Could not write images")
        
        with open(data_idx_cfg_dir, "w") as f:
            count += 1
            f.write(str(count))

### 8. 启动多线程

In [10]:
thread1 = threading.Thread(name='t1',target=capture_video, args=())
thread1.start()

thread2 = threading.Thread(name='t2',target=button_control, args=())
thread2.start()

thread3 = threading.Thread(name='t3',target=save_image, args=())
thread3.start()

Image(value=b'', format='jpg', height='480', width='640')

1
Images-2023-10-25-07_11_16/1.jpg
(480, 640, 3)
<class 'numpy.ndarray'>
